In [6]:
# ============================================================
# Class Activity-6: Food Delivery Chatbot (Machine Learning)
# BSSE23 & BSCE22 | SE305T & MD445T (Spring-26)
# Dr. Muhammad Asif | ITU Lahore
# ============================================================

# ----------------------------------------------------------
# Task 1: Install & Import Required Libraries
# ----------------------------------------------------------
# Run this in Google Colab if not already installed:
# !pip install nltk scikit-learn numpy

import nltk
import numpy as np
import random
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder

nltk.download('punkt', quiet=True)


# ----------------------------------------------------------
# Task 2: Preprocessing Function
# ----------------------------------------------------------
def preprocess(text):
    """Converts text to lowercase and removes extra spaces."""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# ----------------------------------------------------------
# Task 3: Define Chatbot Intents
# ----------------------------------------------------------
intents = {
    "greeting": {
        "patterns": [
            "hello", "hi", "hey", "good morning", "good evening",
            "howdy", "what's up", "hi there", "greetings", "sup"
        ],
        "responses": [
            "Hello! Welcome to QuickBite Food Delivery! How can I help you today?",
            "Hi there! Ready to order some delicious food?",
            "Hey! Great to see you. What can I get for you today?",
            "Welcome! I'm your food delivery assistant. How may I assist you?"
        ]
    },

    "order_food": {
        "patterns": [
            "I want to order food", "place an order", "I'd like to order",
            "can I order", "order pizza", "order burger", "I want food",
            "order something to eat", "I want to place an order",
            "can you take my order", "order now", "I want to buy food"
        ],
        "responses": [
            "Sure! Please browse our menu and tell me what you'd like to order.",
            "Great choice! What would you like to order today?",
            "I'd be happy to take your order! What are you craving?",
            "Let's get that order in! What would you like to eat?"
        ]
    },

    "menu": {
        "patterns": [
            "show menu", "what's on the menu", "what do you have",
            "what food is available", "show me your menu", "list of food",
            "what can I order", "what are your items", "food options",
            "see menu", "menu please", "what dishes do you offer"
        ],
        "responses": [
            "🍕 Our Menu:\n  - Pizza (Margherita, BBQ Chicken) - Rs.800\n  - Burgers (Classic, Zinger) - Rs.450\n  - Pasta (Alfredo, Arrabbiata) - Rs.600\n  - Biryani (Chicken, Beef) - Rs.500\n  - Drinks (Soft drinks, Juices) - Rs.150",
            "Here's what we're serving today:\n  🍔 Burgers | 🍕 Pizza | 🍝 Pasta | 🍚 Biryani | 🥤 Drinks\n  Type an item name to order!",
            "We have a wide variety! Our popular items are Biryani, Zinger Burger, BBQ Pizza, and Alfredo Pasta. Shall I add something to your cart?"
        ]
    },

    "delivery_time": {
        "patterns": [
            "how long will delivery take", "delivery time", "when will my order arrive",
            "how fast is delivery", "estimated delivery", "how long does it take",
            "when will I get my food", "delivery duration", "time for delivery",
            "how many minutes", "ETA", "how soon can you deliver"
        ],
        "responses": [
            "Our standard delivery time is 30–45 minutes depending on your location.",
            "You can expect your order in 30–45 minutes! We'll send you a tracking link once it's dispatched.",
            "Delivery usually takes 30–45 minutes. During peak hours it may take up to 60 minutes.",
            "We aim to deliver within 30–45 minutes. Your food will be fresh and hot!"
        ]
    },

    "payment": {
        "patterns": [
            "how can I pay", "payment methods", "do you accept cash",
            "can I pay online", "credit card payment", "payment options",
            "how to pay", "do you take card", "easypaisa", "jazzcash",
            "COD", "cash on delivery", "payment mode"
        ],
        "responses": [
            "We accept: 💵 Cash on Delivery | 💳 Credit/Debit Cards | 📱 EasyPaisa | 📱 JazzCash",
            "You can pay via Cash on Delivery, Credit/Debit Card, EasyPaisa, or JazzCash. Choose what's convenient!",
            "Payment options: Cash on Delivery (COD), Online Banking, EasyPaisa, JazzCash, and all major credit cards."
        ]
    },

    "contact": {
        "patterns": [
            "contact support", "customer service", "help", "I have a problem",
            "talk to human", "contact number", "support", "reach you",
            "phone number", "email address", "how to contact", "complaint"
        ],
        "responses": [
            "📞 You can reach us at: 0311-1234567 | 📧 support@quickbite.pk\n  We're available 9AM – 11PM daily.",
            "Need help? Contact our support team:\n  📞 0311-1234567\n  📧 support@quickbite.pk\n  ⏰ Available: 9AM–11PM",
            "Our customer support is here for you! Call 0311-1234567 or email support@quickbite.pk."
        ]
    }
}


# ----------------------------------------------------------
# Task 4: Prepare Training Data
# ----------------------------------------------------------
training_patterns = []
training_intents = []

for intent_name, intent_data in intents.items():
    for pattern in intent_data["patterns"]:
        training_patterns.append(preprocess(pattern))
        training_intents.append(intent_name)

print(f"[INFO] Total training samples: {len(training_patterns)}")
print(f"[INFO] Intents: {list(intents.keys())}")


# ----------------------------------------------------------
# Task 5: Feature Extraction using TF-IDF
# ----------------------------------------------------------
vectorizer = TfidfVectorizer(ngram_range=(1, 2), analyzer='word')  # unigram + bigram
X_train = vectorizer.fit_transform(training_patterns)

print(f"[INFO] Feature matrix shape: {X_train.shape}")


# ----------------------------------------------------------
# Task 6: Train Multinomial Naive Bayes Classifier
# ----------------------------------------------------------
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(training_intents)

model = MultinomialNB()
model.fit(X_train, y_train)

print("[INFO] Model trained successfully!")


# ----------------------------------------------------------
# Task 7: Build Prediction Function
# ----------------------------------------------------------
def predict_intent(user_input):
    """
    Predicts the intent from user input.
    Returns: (predicted_intent, confidence_score)
    """
    processed = preprocess(user_input)
    features = vectorizer.transform([processed])

    predicted_label = model.predict(features)[0]
    predicted_intent = label_encoder.inverse_transform([predicted_label])[0]

    # Confidence score from predict_proba
    probabilities = model.predict_proba(features)[0]
    confidence = round(float(np.max(probabilities)), 4)

    return predicted_intent, confidence


# ----------------------------------------------------------
# Task 8: Build Response Function
# ----------------------------------------------------------
CONFIDENCE_THRESHOLD = 0.15

def get_response(user_input):
    """
    Generates a chatbot response based on predicted intent and confidence.
    """
    intent, confidence = predict_intent(user_input)

    if confidence < CONFIDENCE_THRESHOLD:
        return (
            "I'm not quite sure I understood that. 🤔\n"
            "You can ask me about: menu, ordering food, delivery time, payment, or contact support."
        )

    responses = intents[intent]["responses"]
    return random.choice(responses)


# ----------------------------------------------------------
# Task 9: Interactive Chatbot Loop
# ----------------------------------------------------------
def run_chatbot():
    print("\n" + "="*55)
    print("   🍔  Welcome to QuickBite Food Delivery Chatbot  🍕")
    print("="*55)
    print("   Type 'quit' to exit the chatbot.\n")

    while True:
        user_input = input("You: ").strip()

        if not user_input:
            continue

        if user_input.lower() == "quit":
            print("Bot: Thank you for using QuickBite! Have a great day! 👋")
            break

        response = get_response(user_input)
        print(f"Bot: {response}\n")


# ----------------------------------------------------------
# Run the chatbot
# ----------------------------------------------------------
if __name__ == "__main__":
    run_chatbot()

[INFO] Total training samples: 71
[INFO] Intents: ['greeting', 'order_food', 'menu', 'delivery_time', 'payment', 'contact']
[INFO] Feature matrix shape: (71, 194)
[INFO] Model trained successfully!

   🍔  Welcome to QuickBite Food Delivery Chatbot  🍕
   Type 'quit' to exit the chatbot.

You: Hi
Bot: Hello! Welcome to QuickBite Food Delivery! How can I help you today?

You: I want to order food
Bot: Let's get that order in! What would you like to eat?

You: show me the menu
Bot: 🍕 Our Menu:
  - Pizza (Margherita, BBQ Chicken) - Rs.800
  - Burgers (Classic, Zinger) - Rs.450
  - Pasta (Alfredo, Arrabbiata) - Rs.600
  - Biryani (Chicken, Beef) - Rs.500
  - Drinks (Soft drinks, Juices) - Rs.150

You: how can I pay
Bot: Payment options: Cash on Delivery (COD), Online Banking, EasyPaisa, JazzCash, and all major credit cards.

You: do you accept cash
Bot: We accept: 💵 Cash on Delivery | 💳 Credit/Debit Cards | 📱 EasyPaisa | 📱 JazzCash

You: do you take jazzcash
Bot: We accept: 💵 Cash on Deliver